# מעבדה 10 — עכבה, החזרה והעברה

כל גל בקורס הזה מגיע בסופו של דבר למקום שבו אינו יכול פשוט להמשיך. חבל קשור לקיר, כבל מגיע למקלט,
אור פוגש זכוכית. המעבדה הזאת לוקחת את הגרסה הפשוטה ביותר של זה — מיתר אחד המחובר לאחר — ומחלצת ממנה
את האלגברה שעוד תהיה בשימוש כשהגלים יהיו אלקטרומגנטיים והגבול יהיה ציפוי של עדשה.

תגלו איך תווך מרגיש לדחיפה, תראו קיר וקצה חופשי מתגלים כאותה נוסחה בשני קצותיה, תמדדו את המשרעות
המוחזרת והעוברת על פני טווח צפיפויות של פי 625, תשקלו את האנרגיה בשני צדי צומת, ותצמידו סרגל שגיאה
למקדם החזרה. החלק האחרון בונה צומת שאינו מהדהד כלל, ועל מיתר זה דורש תחבולה.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לחזות, כתבו את תחזיתכם בתא המיועד **לפני** שאתם מריצים את
התא הבא.

## מפרט המודל

| | |
|---|---|
| **המערכת** | שני מיתרים אידאליים אינסופיים למחצה בצפיפויות $\mu_1$, $\mu_2$ תחת מתיחות אחת $T$, המחוברים בנקודה — מבחינה נומרית רשת אחת עם קפיצה ב-$\mu$ |
| **הדינמיקה** | $\mu\,y_{tt} = T\,y_{xx}$ בכל צד; בצומת, רציפות $y$ ורציפות הכוח הרוחבי $-T\,y_x$ |
| **תנאי השפה** | הצומת; הקצוות החיצוניים קבועים או חופשיים ומוחזקים מחוץ להישג ידה של כל מדידה |
| **האנסמבל** | דטרמיניסטי; רעש גלאי, במקום שבו הוא מתווסף, הוא גאוסי, בלתי תלוי וזרוע בזרע ידוע |
| **מה שהוזנח** | מסת הצומת, קשיחותו והנחתה בו; אובדן מכל סוג; פגיעה אלכסונית, שאין לה משמעות בממד אחד |
| **תחום התוקף** | שיפועים קטנים, $\lvert y_x \rvert \ll 1$, בשני הצדדים, והצומת הוא נקודה — צפיפות המשתנה בהדרגה היא בעיה אחרת, ואוסף התרגילים הופך אותה לכזאת |
| **אופני כישלון** | קריאת $t > 1$ כרווח אנרגטי; החלת מקדמי המשרעת על הספק; שכחת גורם השטף $Z_2/Z_1$ |

כל הפיזיקה נמצאת ב-`wavelab.waves` — פתחו וקראו.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import measurement, waves
from wavelab.validation import seed_study

TENSION = 4.0  # N
MU = 0.01  # kg/m, the first medium's mass per unit length
SPEED = waves.wave_speed(TENSION, MU)  # sqrt(T/mu)
IMPEDANCE = waves.impedance(TENSION, MU)  # sqrt(T mu) = mu v
MM = 1e3  # metres to millimetres, for plotting only

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2026)

print(f"v = sqrt(T/mu)  = {SPEED} m/s")
print(f"Z = sqrt(T mu)  = {IMPEDANCE} kg/s")
print(f"Z = T / v       = {TENSION / SPEED} kg/s   (the same number, asked differently)")

## חלק 1 — איך התווך מרגיש

אחזו בקצהו של מיתר ארוך מאוד ונענעו אותו. כל מה שאתם משגרים מתרחק ואינו חוזר, ולכן ליד היד שלכם
המיתר הוא גל ימני טהור ומקיים את $y_t = -v\,y_x$. הכוח הרוחבי שעליכם לספק הוא אז

$$
F = -T\,y_x = \frac{T}{v}\,y_t = Z\,y_t,
\qquad
Z = \sqrt{T\mu} = \mu v .
$$

כוח פרופורציוני למהירות ו*בפאזה* אתה — המיתר מתנגד כמו בולם, לא כמו קפיץ ולא כמו מסה. למטה נקראים
הכוח והמהירות הרוחבית בכל נקודה של פולס מתקדם ומשורטטים זה מול זה. אם הטענה נכונה, ענן הנקודות הוא
קו ישר שהשיפוע שלו הוא $Z$.

In [ ]:
LENGTH, CELLS = 4.0, 1600
dx = LENGTH / CELLS
x = np.arange(CELLS + 1) * dx
WIDTH, START, HEIGHT = 0.2, 1.0, 0.008


def gaussian(s, centre=START, width=WIDTH, height=HEIGHT):
    """A hump `height` tall and `width` wide (at 1/e), centred at `centre`."""
    return height * np.exp(-(((s - centre) / width) ** 2))


# A pure right-mover starts with y_t = -v y_x; flipping that sign launches it the other way.
shape = gaussian(x)
slope0 = -2.0 * (x - START) / WIDTH**2 * shape
dt = 0.5 * dx / SPEED
right = waves.simulate_string(shape, -SPEED * slope0, dx, dt, TENSION, MU, 400, save_every=20)
left = waves.simulate_string(shape, +SPEED * slope0, dx, dt, TENSION, MU, 400, save_every=20)

snapshot = right.times.size // 2
fig, (string_axis, dashpot_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
string_axis.plot(x, MM * right.y[snapshot], color="0.15")
string_axis.set_xlabel("x (m)"), string_axis.set_ylabel("y (mm)")

dashpot = {}
for run, colour, label in ((right, "tab:blue", "right-mover"), (left, "tab:orange", "left-mover")):
    velocity = run.dydt[snapshot]
    force = -TENSION * np.gradient(run.y[snapshot], dx)
    live = np.abs(velocity) > 0.05 * np.abs(velocity).max()
    dashpot_axis.plot(velocity[live], force[live], ".", ms=3, color=colour, label=label)
    dashpot[label] = float(np.polyfit(velocity[live], force[live], 1)[0])
    print(f"{label:12s}: fitted F / (dy/dt) = {dashpot[label]:+.6f} kg/s")

dashpot_axis.set_xlabel("transverse velocity dy/dt (m/s)")
dashpot_axis.set_ylabel("transverse force -T dy/dx (N)")
dashpot_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Z = sqrt(T mu)               = {IMPEDANCE:+.6f} kg/s")
print()
print("A right-mover meets +Z and a left-mover -Z: the sign is the direction the energy goes,")
print("not a different medium. Nothing about the pulse's height or width enters the slope.")

## חלק 2 — נסו לחזות

התחייבו לתשובה על כל אחת לפני שתריצו משהו למטה.

1. פולס נע לאורך חבל הקשור לקיר. הוא חוזר ישר או הפוך?
2. אותו פולס נע לאורך חבל כבד הקשור בקשר לחוט קל וממשיך אליו. האם הפולס ה*מוחזר* ישר או הפוך, והאם
   הפולס ה*עובר* גבוה או נמוך מן הפוגע?
3. האם הפולס העובר יכול להיות גבוה מן הפולס שיצר אותו? אם כן, מהיכן מגיעה האנרגיה הנוספת?
4. האם יש צומת בין שני מיתרים שונים שאינו מייצר הד כלל?

**התחזית שלכם:**

## חלק 3 — קיר וקצה חופשי

לפותר יש שני תנאי שפה ואף אחד מהם אינו מזכיר עכבה. קצה `"fixed"` מוחזק ב-$y = 0$; קצה `"free"` הוא
טבעת חסרת מסה על מוט חסר חיכוך, שמחליקה כך שהשיפוע שם הוא אפס.

ראו מה כל אחד מהם עושה לאותו פולס, והשוו ל-$r = (Z_1 - Z_2)/(Z_1 + Z_2)$ בשני גבולותיו:
$Z_2 \to \infty$ לקיר, $Z_2 \to 0$ לקצה חופשי.

In [ ]:
END_LENGTH, END_CELLS = 2.0, 1200
end_dx = END_LENGTH / END_CELLS
end_x = np.arange(END_CELLS + 1) * end_dx
END_START, END_WIDTH = 0.7, 0.15

end_shape = gaussian(end_x, END_START, END_WIDTH)
end_slope = -2.0 * (end_x - END_START) / END_WIDTH**2 * end_shape
end_dt = 0.5 * end_dx / SPEED
end_steps = int(round(2.3 / SPEED / end_dt))

ends = {}
fig, axes = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True)
for axis, far_end, colour in ((axes[0], "fixed", "tab:blue"), (axes[1], "free", "tab:orange")):
    run = waves.simulate_string(
        end_shape, -SPEED * end_slope, end_dx, end_dt, TENSION, MU, end_steps,
        boundary=("free", far_end), save_every=max(1, end_steps // 120),
    )
    returned = run.y[-1][np.argmax(np.abs(run.y[-1]))] / HEIGHT
    energy = waves.total_energy(run.y, run.dydt, end_dx, TENSION, MU)
    ends[far_end] = (float(returned), float(run.y[:, -1].max() / HEIGHT),
                     float(abs(energy[-1] / energy[0] - 1.0)))

    axis.plot(end_x, MM * run.y[0], color="0.8", lw=1.0, ls=":")
    axis.plot(end_x, MM * run.y[-1], color=colour, lw=1.6)
    axis.axvline(END_LENGTH, color="0.2", lw=2.5 if far_end == "fixed" else 1.0)
    axis.set_ylabel("y (mm)")
    print(f"{far_end:5s} end: pulse returns at {returned:+.4f} A, the end itself reaches "
          f"{run.y[:, -1].max() / HEIGHT:+.4f} A, energy kept to "
          f"{abs(energy[-1] / energy[0] - 1):.1e}")
axes[1].set_xlabel("x (m)")
plt.tight_layout()
plt.show()

print()
print("A wall returns -1 and a free end +1, and neither takes any energy. The dotted curve is")
print("where the pulse started; the solid one is the same pulse on its way back.")

## חלק 4 — צומת אחד, משני צדדיו

עכשיו חברו שני מיתרים במקום זאת. הרשת היא מיתר אחד שצפיפותו קופצת בנקודה, וזה כל מה שצומת הוא:
`simulate_string` מקבל מערך `mu`, ושני תנאי ההתאמה — המיתר אינו קרוע, המפרק חסר מסה — אינם נכפים
בשום מקום. הם תוצאה של כך שהסכמה מעדכנת כל נקודה משכנותיה.

שני הפאנלים משתמשים ביחס צפיפויות 4:1, ולכן העכבות נבדלות פי שתיים והמקדמים הם בדיוק שלישים. רק
כיוון ההגעה משתנה.

In [ ]:
APPROACH = 1.2  # m of the first medium crossed before the junction is reached
CLEARANCE = 3.0  # pulse widths of separation insisted on before anything is measured


def junction_run(ratio, points_per_width=20, amplitude=HEIGHT, width=WIDTH):
    """Fire a right-moving Gaussian at a jump to `ratio` times the first medium's density.

    The geometry is scaled to the two wave speeds. A transmitted pulse is stretched by
    v_2/v_1, so the far side is made that much longer to hold it and the grid fine enough to
    resolve whichever of the two pulses is narrower.
    """
    speed_2 = waves.wave_speed(TENSION, MU * ratio)
    width_2 = width * speed_2 / SPEED
    step = min(width, width_2) / points_per_width
    clear = CLEARANCE * width / SPEED

    junction = APPROACH + SPEED * clear + 2.0 * width
    length = junction + speed_2 * clear + 2.0 * width_2
    cells = int(round(length / step))
    step = length / cells

    grid = np.arange(cells + 1) * step
    mu = np.where(grid < junction, MU, MU * ratio)
    centre = junction - APPROACH
    start = gaussian(grid, centre, width, amplitude)
    start_slope = -2.0 * (grid - centre) / width**2 * start

    dt = 0.5 * step / max(SPEED, speed_2)
    run = waves.simulate_string(
        start, -SPEED * start_slope, step, dt, TENSION, mu,
        int(round((APPROACH / SPEED + clear) / dt)),
    )
    return run, step, mu, junction


def split_amplitudes(run, dx_run, junction, ratio, amplitude=HEIGHT, width=WIDTH):
    """Signed reflected and transmitted amplitudes, as fractions of the incident one."""
    final = run.y[-1]
    width_2 = width * waves.wave_speed(TENSION, MU * ratio) / SPEED

    def extremum(mask):
        segment = final[mask]
        return float(segment[np.argmax(np.abs(segment))] / amplitude)

    return extremum(run.x < junction - width), extremum(run.x > junction + width_2)


pair = {}
fig, axes = plt.subplots(2, 1, figsize=(10, 4.4))
for axis, ratio, colour in ((axes[0], 0.25, "tab:blue"), (axes[1], 4.0, "tab:orange")):
    run, step, mu, junction = junction_run(ratio)
    r_measured, t_measured = split_amplitudes(run, step, junction, ratio)
    r, t = waves.junction_coefficients(IMPEDANCE, waves.impedance(TENSION, MU * ratio))
    pair[ratio] = (r, r_measured, t, t_measured)

    # the shading marks the heavier medium, whichever side of the junction it is on
    heavier = (junction, run.x[-1]) if ratio > 1.0 else (0.0, junction)
    axis.axvspan(*heavier, color="0.5", alpha=0.12, lw=0.0)
    axis.plot(run.x, MM * run.y[-1], color=colour, lw=1.5)
    axis.axvline(junction, color="0.35", lw=1.0, ls="--")
    axis.set_xlim(0.0, run.x[-1]), axis.set_ylabel("y (mm)")
    print(f"mu2/mu1 = {ratio:5.2f}:  r = {r:+.4f} (measured {r_measured:+.4f}),  "
          f"t = {t:.4f} (measured {t_measured:.4f})")
axes[1].set_xlabel("x (m)")
plt.tight_layout()
plt.show()

print()
print("Onto the lighter string the echo is upright and the transmitted pulse is taller and")
print("wider; onto the heavier one the echo is inverted and the transmitted pulse is shorter")
print("and narrower. One junction, two directions, and the sign of r is what changed.")

## חלק 5 — הסריקה

זוג הרצות אחד הוא אנקדוטה. למטה חוזרים על אותה מדידה על פני טווח צפיפויות של פי 625 ומניחים אותה על
שתי הנוסחאות

$$
r = \frac{Z_1 - Z_2}{Z_1 + Z_2},
\qquad
t = \frac{2 Z_1}{Z_1 + Z_2},
\qquad
Z = \sqrt{T\mu} .
$$

שימו לב היכן $r$ מחליף סימן, ושימו לב ש-$t$ לעולם אינו מחליף.

In [ ]:
ratios = np.array([0.04, 0.0625, 0.111, 0.25, 0.5, 1.0, 2.0, 4.0, 9.0, 16.0, 25.0])
measured_r, measured_t = [], []
for ratio in ratios:
    run, step, mu, junction = junction_run(ratio)
    reflected, transmitted = split_amplitudes(run, step, junction, ratio)
    measured_r.append(reflected)
    measured_t.append(transmitted)

impedances = waves.impedance(TENSION, MU * ratios)
predicted_r, predicted_t = waves.junction_coefficients(IMPEDANCE, impedances)

smooth = np.logspace(np.log10(ratios[0]), np.log10(ratios[-1]), 200)
curve_r, curve_t = waves.junction_coefficients(IMPEDANCE, waves.impedance(TENSION, MU * smooth))

plt.figure(figsize=(7.5, 3.6))
plt.semilogx(smooth, curve_r, color="tab:blue", lw=1.2, label="r, formula")
plt.semilogx(smooth, curve_t, color="tab:orange", lw=1.2, label="t, formula")
plt.semilogx(ratios, measured_r, "o", ms=5, color="tab:blue", label="r, measured")
plt.semilogx(ratios, measured_t, "s", ms=5, color="tab:orange", label="t, measured")
plt.axhline(0.0, color="0.85", lw=0.8)
plt.axvline(1.0, color="0.85", lw=0.8)
plt.xlabel("mu2 / mu1"), plt.ylabel("amplitude ratio")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("mu2/mu1      r (formula)   r (measured)    t (formula)   t (measured)")
for ratio, r, rm, t, tm in zip(ratios, predicted_r, measured_r, predicted_t, measured_t):
    print(f"{ratio:8.3f}      {r:+.4f}       {rm:+.4f}        {t:.4f}        {tm:.4f}")
print()
print(f"worst disagreement over the sweep: "
      f"{max(np.max(np.abs(np.array(measured_r) - predicted_r)), np.max(np.abs(np.array(measured_t) - predicted_t))):.1e}")
print("r changes sign exactly at mu2 = mu1. Inversion is not what reflection does; it is what")
print("reflection does off something harder.")

## חלק 6 — ספרי האנרגיה

יחס משרעות הגדול מ-1 נראה כמו משהו יש מאין, אז שקלו את האנרגיה במקום. הפולס הפוגע שווה יחידה אחת;
שתי החתיכות שהוא הופך אליהן נשקלות עם `total_energy` בשני צדי הצומת, ומושוות אל

$$
R = r^2,
\qquad
T = \frac{Z_2}{Z_1}\,t^2,
\qquad
R + T = 1 .
$$

הגורם $Z_2/Z_1$ הוא כל ההבדל בין משרעת להספק, והוא הסיבה לכך שפולס עובר הגבוה פי שניים יכול לשאת
כמעט כלום.

In [ ]:
book_ratios = np.array([0.04, 0.25, 1.0, 4.0, 25.0])
weighed_R, weighed_T = [], []
for ratio in book_ratios:
    run, step, mu, junction = junction_run(ratio)
    cut = int(round(junction / step))
    started = waves.total_energy(run.y[0], run.dydt[0], step, TENSION, mu)
    behind = waves.total_energy(
        run.y[-1][: cut + 1], run.dydt[-1][: cut + 1], step, TENSION, mu[: cut + 1]
    )
    beyond = waves.total_energy(run.y[-1][cut:], run.dydt[-1][cut:], step, TENSION, mu[cut:])
    weighed_R.append(behind / started)
    weighed_T.append(beyond / started)

predicted_R, predicted_T = waves.power_coefficients(
    IMPEDANCE, waves.impedance(TENSION, MU * book_ratios)
)

positions = np.arange(len(book_ratios))
plt.figure(figsize=(7.5, 3.2))
plt.bar(positions - 0.17, weighed_R, 0.32, color="tab:blue", label="R weighed")
plt.bar(positions + 0.17, weighed_T, 0.32, color="tab:orange", label="T weighed")
plt.plot(positions - 0.17, predicted_R, "k_", ms=14, label="formula")
plt.plot(positions + 0.17, predicted_T, "k_", ms=14)
plt.xticks(positions, [f"{ratio:g}" for ratio in book_ratios])
plt.xlabel("mu2 / mu1"), plt.ylabel("fraction of the incident energy")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("mu2/mu1    R (formula)  R (weighed)   T (formula)  T (weighed)   R + T")
for ratio, R, Rw, T, Tw in zip(book_ratios, predicted_R, weighed_R, predicted_T, weighed_T):
    print(f"{ratio:7.2f}      {R:.6f}     {Rw:.6f}      {T:.6f}     {Tw:.6f}   {Rw + Tw:.8f}")
print()
print("Two things to notice. Nothing is lost — no mechanism for loss was put in, so a leak")
print("would have been the scheme's. And the rows for mu2/mu1 = 1/4 and 4 are identical:")
print("the same fraction comes back whichever side the pulse arrives from, though the")
print("amplitudes in the two cases are completely different.")

## חלק 7 — מקדם החזרה עם סרגל שגיאה

עד כה כל מספר הגיע מרשומה נטולת רעש. מדידה אמיתית של $r$ פירושה לצלם חבל ולקרוא שתי משרעות מפריימים
מגורענים, ולכן כאן הפריים האחרון מזוהם ברעש גאוסי בגודל 5% מן המשרעת הפוגעת והמשרעת נאמדת פעמיים
מאותם נתונים:

- ב**היטל** על צורת הפולס שהגאומטריה חוזה — אומד לינארי, ולכן הרעש מתמצע החוצה ממנו;
- ב**בחירת הסטייה הגדולה ביותר** בחלון, וזה מה שהעין עושה.

שניהם חשבון ישר. רק אחד מהם הוא מדידה של מקדם ההחזרה.

In [ ]:
SIGMA = 0.05 * HEIGHT  # 5% of the incident amplitude, independent at every point
SEEDS = 64
noisy_r = {}

print(f"noise: sigma = {SIGMA * MM:.2f} mm on an incident pulse {HEIGHT * MM:.0f} mm tall, "
      f"{SEEDS} seeds each")
print()
print("mu2/mu1       r      noiseless      projection            peak-pick"
      "        per-frame sigma")
for ratio in (0.25, 1.0, 4.0):
    run, step, mu, junction = junction_run(ratio)
    # the reflected pulse has travelled back exactly CLEARANCE pulse widths from the junction
    echo_centre = junction - CLEARANCE * WIDTH
    template = gaussian(run.x, echo_centre, WIDTH, 1.0)
    window = np.abs(run.x - echo_centre) < 2.0 * WIDTH
    frame = run.y[-1]
    r, _ = waves.junction_coefficients(IMPEDANCE, waves.impedance(TENSION, MU * ratio))

    def project(generator, frame=frame, template=template):
        """Least-squares amplitude of the expected pulse shape — linear in the data."""
        noisy = measurement.add_noise(frame, SIGMA, generator)
        return float(noisy @ template / (template @ template)) / HEIGHT

    def pick(generator, frame=frame, window=window):
        """The largest excursion in the window — what an eye does with a grainy film."""
        noisy = measurement.add_noise(frame, SIGMA, generator)[window]
        return float(noisy[np.argmax(np.abs(noisy))]) / HEIGHT

    clean = float(frame @ template / (template @ template)) / HEIGHT
    projection = seed_study(project, n_seeds=SEEDS, base_seed=10)
    peak = seed_study(pick, n_seeds=SEEDS, base_seed=10)
    noisy_r[ratio] = (r, clean, projection, peak)

    expected_sigma = SIGMA / np.sqrt(template @ template) / HEIGHT
    print(f"{ratio:7.2f}   {r:+.4f}  {clean:+.4f}   {projection.mean:+.4f} +/- "
          f"{projection.standard_error:.4f}   {peak.mean:+.4f} +/- "
          f"{peak.standard_error:.4f}    {np.std(projection.values, ddof=1):.4f} "
          f"(predicted {expected_sigma:.4f})")

print()
print("On the noiseless frame the projection reproduces r to better than 0.1%, so the")
print("estimator itself is sound. Add the noise and each single frame scatters by about")
print("0.011 - close to the sigma/sqrt(sum of squares) the noise level predicts - while the")
print("mean of 64 frames stays within a couple of standard errors of the noiseless value.")
print("That is what a linear estimator buys: the noise averages out instead of piling up.")
print()
print("The peak-pick is biased away from zero by about a fifth of the answer, and no number")
print("of frames repairs it: the largest of many noisy samples is not an estimate of the")
print("signal at any of them, it is the noise's best excursion handed to you. This is")
print("module 09's biased wattmeter wearing a different coat, and it is why part 8 hunts")
print("the match by weighing energy rather than by looking for a pulse that is not there.")

## חלק 8 — מצאו את התיאום

כווננו את צפיפות המיתר השני עד שההד נעלם. הקריאה היא שבר האנרגיה שחזר, וזה הגודל שמדידה יכולה
באמת לדחוף לאפס.

על מיתר זהו כלי גס: המתיחות היא מספר אחד מקצה לקצה, ולכן עכבה שווה פירושה צפיפות שווה, והתיאום
שאתם צדים הוא $\mu_2 = \mu_1$ — אותו מיתר. שימו לב בכל זאת לצורת העקומה בהתקרבכם אליו. $R$ הוא
*ריבועי* באי ההתאמה בסביבת התיאום: סטייה של 10% בצפיפות עולה כ-0.06% מן האנרגיה. תיאום הוא סלחני,
ולכן כדאי לעשות אותו בקירוב גם בעולם האמיתי.

In [ ]:
import ipywidgets as widgets


def echo(density_ratio=4.0):
    run, step, mu, junction = junction_run(density_ratio, points_per_width=14)
    cut = int(round(junction / step))
    started = waves.total_energy(run.y[0], run.dydt[0], step, TENSION, mu)
    behind = waves.total_energy(
        run.y[-1][: cut + 1], run.dydt[-1][: cut + 1], step, TENSION, mu[: cut + 1]
    )
    r, t = waves.junction_coefficients(IMPEDANCE, waves.impedance(TENSION, MU * density_ratio))
    reflected, transmitted = waves.power_coefficients(
        IMPEDANCE, waves.impedance(TENSION, MU * density_ratio)
    )

    plt.figure(figsize=(9, 2.6))
    plt.axvspan(junction, run.x[-1], color="0.5", alpha=0.12, lw=0.0)
    plt.plot(run.x, MM * run.y[-1], color="tab:blue", lw=1.4)
    plt.axvline(junction, color="0.35", lw=1.0, ls="--")
    plt.xlim(0.0, run.x[-1]), plt.ylim(-12.0, 16.0)
    plt.xlabel("x (m)"), plt.ylabel("y (mm)")
    plt.tight_layout()
    plt.show()

    print(f"Z2 / Z1                 : {np.sqrt(density_ratio):.4f}")
    print(f"r, t                    : {r:+.4f}, {t:.4f}")
    print(f"R (formula / weighed)   : {reflected:.5f} / {float(behind / started):.5f}")
    print(f"T (formula)             : {transmitted:.5f}")


widgets.interact_manual(
    echo,
    density_ratio=widgets.FloatLogSlider(
        base=10, min=-1.4, max=1.4, step=0.05, value=4.0, description="mu2/mu1"
    ),
);

## חלק 9 — מתקדם: צומת שבאמת אינו מהדהד

תיאום בין שני תווכים *נתונים* הוא הבעיה השימושית, ועל מיתר אי אפשר לפתור אותה בבחירת $\mu_2$: שני
התווכים נתונים. אפשר לפתור אותה בהכנסת שלישי.

שימו ביניהם מקטע שעכבתו $Z_2 = \sqrt{Z_1 Z_3}$ ואורכו רבע אורך גל כפי שנמדד *במקטע עצמו*. שתי
פאותיו מחזירות כל אחת גל קטן; הם שווים בגודלם ורחוקים חצי אורך גל זה מזה בהלוך ושוב, ולכן הם
מתבטלים. התא למטה שולח את אותה חבילה אל מדרגת צפיפות 9:1 חשופה (מדרגת עכבה 3:1) ואל אותה מדרגה
כשמותקן בה הפס הזה.

זהו ציפוי נוגד החזרה, שני חלקים לפני שהקורס מגיע להתאבכות. מודול 24 עושה זאת עם אור ועם עדשה, שם
הפס הוא כמה מאות ננומטרים של מגנזיום פלואוריד.

In [ ]:
WAVELENGTH = 0.4  # m, in the first medium
DENSITY_STEP = 9.0  # mu3 / mu1, so Z3 / Z1 = 3
FREQUENCY = SPEED / WAVELENGTH

match_length, match_cells = 8.0, 1600
match_dx = match_length / match_cells
match_x = np.arange(match_cells + 1) * match_dx
JUNCTION = 5.0

section_density = np.sqrt(DENSITY_STEP) * MU  # Z2 = sqrt(Z1 Z3)
section_length = 0.25 * waves.wave_speed(TENSION, section_density) / FREQUENCY


def packet(grid, centre, cycles=4.0):
    """A sine of `cycles` periods under a cos^2 envelope — a train with a beginning and an end."""
    span = cycles * WAVELENGTH
    envelope = np.where(
        np.abs(grid - centre) < 0.5 * span, np.cos(np.pi * (grid - centre) / span) ** 2, 0.0
    )
    return 0.004 * envelope * np.sin(2.0 * np.pi * (grid - centre) / WAVELENGTH)


bare = np.where(match_x < JUNCTION, MU, DENSITY_STEP * MU)
matched = np.where(
    (match_x >= JUNCTION) & (match_x < JUNCTION + section_length), section_density, bare
)

start = packet(match_x, JUNCTION - 2.5)
match_dt = 0.5 * match_dx / SPEED
match_steps = int(round(5.0 / SPEED / match_dt))
cut = int(round(JUNCTION / match_dx))

echoes = {}
fig, axes = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True)
for axis, mu, name, colour in (
    (axes[0], bare, "bare step", "tab:blue"),
    (axes[1], matched, "with lambda/4 section", "tab:orange"),
):
    run = waves.simulate_string(
        start, -SPEED * np.gradient(start, match_dx), match_dx, match_dt,
        TENSION, mu, match_steps,
    )
    started = waves.total_energy(run.y[0], run.dydt[0], match_dx, TENSION, mu)
    behind = waves.total_energy(
        run.y[-1][:cut], run.dydt[-1][:cut], match_dx, TENSION, mu[:cut]
    )
    axis.axvspan(JUNCTION, match_length, color="0.5", alpha=0.12, lw=0.0)
    axis.axvspan(JUNCTION, JUNCTION + section_length, color="tab:green", alpha=0.35, lw=0.0)
    axis.plot(match_x, MM * run.y[-1], color=colour, lw=1.2)
    axis.set_ylabel("y (mm)"), axis.set_ylim(-6.0, 6.0)
    echoes[name] = (float(behind / started), np.abs(run.y[-1][: cut - 100]).max() / 0.004)
    print(f"{name:22s}: R = {echoes[name][0]:.4f}, largest echo {echoes[name][1]:.3f} A")
axes[1].set_xlabel("x (m)")
plt.tight_layout()
plt.show()

print()
print(f"the section is {section_length * 1e3:.1f} mm of string three times the first one's")
print(f"density — a quarter of a wavelength at {FREQUENCY:.0f} Hz, measured in the section")
print("itself, where the wave is neither neighbour's length.")
print()
print("Nothing absorbed the missing echo. Two reflections cancelled, which is the first")
print("interference calculation in the course and the reason camera lenses are purple.")

In [ ]:
# 1. The medium is a dashpot of resistance Z, and the sign is the direction of travel.
assert abs(dashpot["right-mover"] - IMPEDANCE) < 1e-3 * IMPEDANCE
assert abs(dashpot["left-mover"] + IMPEDANCE) < 1e-3 * IMPEDANCE

# 2. The two ends are the two limits of one formula, and neither takes any energy.
assert abs(ends["fixed"][0] + 1.0) < 1e-3 and abs(ends["free"][0] - 1.0) < 1e-3
assert abs(ends["free"][1] - 2.0) < 1e-2
assert max(ends["fixed"][2], ends["free"][2]) < 1e-6

# 3. The measured split is the impedances' split, over the whole sweep, and r changes sign
#    exactly at the match — the falsifier for `reflection-always-inverts`.
assert np.max(np.abs(np.array(measured_r) - predicted_r)) < 3e-3
assert np.max(np.abs(np.array(measured_t) - predicted_t)) < 3e-3
assert all(np.sign(m) == np.sign(p) for m, p in zip(measured_r, predicted_r) if p != 0.0)
assert min(measured_t) > 0.0  # t never changes sign, whatever the junction

# 4. The books balance, and they balance the same way from either side.
assert np.max(np.abs(np.array(weighed_R) + np.array(weighed_T) - 1.0)) < 1e-4
assert np.max(np.abs(np.array(weighed_R) - predicted_R)) < 5e-4
assert abs(weighed_R[1] - weighed_R[3]) < 1e-5  # mu2/mu1 = 1/4 and 4 reflect alike

# 5. Through the noise: the projection finds r, the peak-pick finds the noise.
for ratio, (r, clean, projection, peak) in noisy_r.items():
    assert abs(clean - r) < 1e-2           # the estimator's own systematic, before any noise
    assert projection.agrees_with(clean)   # and the noise averages out of it
    if r != 0.0:
        assert not peak.agrees_with(clean)
        assert abs(peak.mean) > 1.15 * abs(clean)

# 6. A quarter-wave section does what no choice of mu2 can do on a string.
assert echoes["bare step"][0] > 0.2
assert echoes["with lambda/4 section"][0] < 0.1 * echoes["bare step"][0]

print("all checks passed")
print(f"reflection coefficient through 5% noise at mu2/mu1 = 4: "
      f"{noisy_r[4.0][2].mean:+.4f} +/- {noisy_r[4.0][2].standard_error:.4f} "
      f"against (Z1 - Z2)/(Z1 + Z2) = {noisy_r[4.0][0]:+.4f}")

## בדקו את הבנתכם

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "10-impedance.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. בחלק 3 קיר וקצה חופשי החזירו את הפולס בסימנים מנוגדים, ובחלק 4 צומת עשה את אותו הדבר בהתאם לצד
   שממנו הגיע הפולס. הסבירו למי שמאמין שהחזרה תמיד הופכת מה באמת קובע את הסימן, בלי לכתוב נוסחה.
2. בחלק 5 המשרעת העוברת הגיעה ל-1.67 בזמן שהמשרעת המוחזרת הייתה 0.67, וחלק 6 הראה ששום דבר לא
   נוסף. אמרו היכן הנהלת החשבונות בולעת פולס עובר הגבוה מן הפוגע.
3. בחלק 7 שני חישובים נכונים על אותו פריים רועש נבדלו בחמישית מן התשובה. אמרו מה הופך אומד למוטה
   ולא סתם לרועש, ונקבו במדידה אחת בהמשך הקורס — של אור, לא של מיתר — שבה אותה מלכודת ממתינה.

**התשובות שלכם:**